# Ambient sound, one week, one minute at a time**What this notebook does**1. Generates a synthetic week of ambient sound data — one room, one value per minute, 10,080 rows.2. Thinks the problem through before drawing anything (§1). The brief has three real ambiguities in it and a couple of traps that are specific to acoustic data.3. Draws the requested chart: every day as a soft grey trace on a midnight-to-midnight axis, with the across-day median in black.4. Adds a day selector so one day can be pulled forward against that reference frame.5. Offers the alternatives that I think are better for some of the questions you'll actually ask of this data.There is a companion standalone HTML viewer (`sound_week_viewer.html`) with the same data and no Python dependency — see §8.Units are `L_Aeq,1min` in dB SPL: the A-weighted equivalent continuous level over each one-minute window, which is what a room-level acoustic monitor typically streams. Descriptive only — nothing here is a diagnostic threshold.

---## 1. Observing the problem### 1.1 What was asked, restated> One panel, x-axis midnight → midnight. All seven days drawn faintly in grey. The median across days drawn strongly in black. A control to choose which day of the week to look at.That is a **focus + context** chart: the grey traces and the median are the *context* (what a normal day looks like here), and the selected day is the *focus* (how this particular day departs from normal). Getting that relationship right is the whole job — everything below follows from it.### 1.2 The ambiguities**"the distribution of all sounds per day"** can mean three different charts, and they answer different questions:| Reading | Chart | Answers ||---|---|---|| **A.** Each day is a curve over time | 7 overlaid line traces | "What does the shape of a day look like, and do the days agree?" || **B.** At each minute, the spread across the 7 days | A percentile band (p10–p90, p25–p75) + median | "What is the *normal range* for 03:14, and is today inside it?" || **C.** Ignoring time, the spread of levels within a day | Histogram / violin / ECDF per day | "How loud was Tuesday overall?" |Your description — grey daily curves plus a black median — is **A with the median of B laid over it**. That's a good default, so it's what §4 builds. But B is the one that generalises: with 7 days a spaghetti plot still reads; with 90 days it becomes a solid grey block, and the band is the only version that survives. §5 builds it, and I'd make it the primary view for anything beyond a fortnight.**"the median for all days"** is also two things. Pointwise median across days *at each minute* (a curve that varies with time of day) — almost certainly what you want, and what's implemented. Not: a single scalar median for the week, which would be a flat line.### 1.3 The traps**Decibels are logarithmic, so arithmetic means lie.** The median is safe — it's rank-based, and a monotone transform doesn't move it, so the median of the dB values *is* the dB of the median sound pressure. Means are not safe. If you ever want an average level, average the energy and convert back:$$L_{eq} = 10\log_{10}\left(\frac{1}{N}\sum_i 10^{L_i/10}\right)$$A one-minute 70 dB cough sitting in an hour of 30 dB silence moves the arithmetic mean by ~0.7 dB and the energy mean by ~12 dB. §7 computes both, deliberately, so the gap is visible. This is the single most common error in room-acoustics dashboards.**Overplotting.** 1440 points per day × 7 days, with impulsive spikes on top of a slow baseline, is 10,080 line segments in one panel. Raw, it's grey mush and the black median is the only readable thing. Options: reduce alpha (helps a bit), bin to 5-minute windows (loses the spikes, which for night monitoring *are the signal*), or smooth with a **rolling median** rather than a rolling mean. I use a 5-minute centred rolling median for display: it keeps the baseline honest, suppresses single-sample sensor noise, and — unlike a mean — doesn't let one 70 dB spike smear across five minutes. The raw series is always kept alongside; smoothing is a display choice, never a data choice.**Midnight-to-midnight splits the night.** You asked for it and it's the right default for a calendar-shaped chart, but it puts the most clinically interesting window — the sleep period — at the two opposite edges of the panel, cut in half. For overnight deterioration work, a **noon-to-noon axis** keeps a night intact in the middle of the frame and makes "night of the 19th" a single visual object. §9 draws it. I'd have both views and let the question pick.**A "day" is not always 1440 minutes.** On DST transition days a local calendar day is 1380 or 1500 minutes long, so a fixed 1440-row matrix silently misaligns. Options: index on minutes-since-local-midnight and accept two ragged days a year, or run the whole pipeline in UTC and only convert for labels. For a UK deployment this bites twice a year, on the last Sundays of March and October. The synthetic week here avoids the transition; the reshape in §3 asserts the shape so it fails loudly rather than quietly.**Missing data.** Charging, non-wear, sensor dropout. A median computed over 3 available days and one computed over 7 look identical on the page but are not the same quantity. §3 computes per-minute coverage and §4 masks any minute backed by fewer than 4 days, so gaps show as gaps rather than as confident lines.**Seven days is a small sample.** The black median is a 7-point median at each minute. It's a reasonable *reference*, not a population norm — its standard error is roughly 1.25σ/√7, so around half a dB where the days agree and several dB where they don't. That distinction matters a lot for the reference-trend framing: a week describes one room; a norm needs many rooms and many weeks. The band in §5 is the honest way to show it, because it displays the disagreement instead of hiding it under a single line.**Colour.** Seven greys at equal weight compete with each other and with the median. So: one grey, low alpha, thin — the days are texture, not seven identified series. Nobody can trace which grey line is Wednesday, and they shouldn't try; that's what the selection is for. The selected day gets contrast and weight, not membership in a rainbow.### 1.4 Decisions- Grey traces stay visible when a day is selected. Selecting **highlights**, it does not **filter** — a day out of context is just a line.- One grey, one black, one accent. Three inks total.- Smoothing is a rolling median, shown as an adjustable control, and stated in the caption.- The median is masked where coverage is thin.- Night hours are shaded, because that's the window this data exists to serve.- Both the midnight-to-midnight and noon-to-noon frames are provided.

---## 2. Setup and synthetic dataThe generator is deliberately not random noise. It models a care-home room: a smooth diurnal background floor, a slower wake-up at weekends, sustained sources (morning care round, lunch, evening television, weekend visitors), short impulsive night events (cough, restlessness, door), correlated drift for HVAC and outside traffic, and two non-wear gaps. Thursday night is deliberately disturbed so that the selector has something worth finding.Every parameter is in one place and the seed is fixed, so you can turn the knobs and watch the chart respond — which is the real test of whether the chart works.

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom matplotlib.ticker import MultipleLocator, FuncFormatterMIN_PER_DAY = 1440DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday",        "Friday", "Saturday", "Sunday"]# three inks, and nothing elseGREY   = "#B9BCC4"   # the six other days: texture, not seriesINK    = "#101215"   # the median: the referenceHILITE = "#C2452D"   # the selected day: the focusNIGHT  = "#2B3A55"   # 22:00-07:00 washplt.rcParams.update({    "figure.dpi": 110,    "axes.grid": True,    "grid.alpha": 0.25,    "grid.linewidth": 0.6,    "axes.spines.top": False,    "axes.spines.right": False,    "font.size": 10,})

In [ ]:
def _diurnal_floor(minutes, weekend=False):    """Background level in dB(A) as a smooth function of time of day."""    t = minutes / MIN_PER_DAY    wake  = 6.75 / 24 + (1.25 / 24 if weekend else 0.0)    sleep = 22.5 / 24 + (0.75 / 24 if weekend else 0.0)    rise = 1 / (1 + np.exp(-(t - wake) * 90))    fall = 1 / (1 + np.exp((t - sleep) * 90))    day_shape = rise * fall    dip = -2.5 * np.exp(-((t - 14.0 / 24) ** 2) / (2 * (0.9 / 24) ** 2))    return 29.5 + 13.0 * day_shape + dip * day_shapedef _ar1(n, rho, sigma, rng):    """Correlated drift: HVAC, weather, traffic outside the window."""    e = rng.normal(0, sigma, n)    x = np.zeros(n)    for i in range(1, n):        x[i] = rho * x[i - 1] + e[i]    return xdef _add_block(level, start_min, dur_min, gain, rng, ramp=4):    """Add a sustained source (television, vacuum, visitors) with soft edges."""    n = len(level)    idx = (np.arange(start_min, start_min + dur_min)) % n    env = np.ones(dur_min)    r = min(ramp, dur_min // 2)    if r > 0:        env[:r]  = np.linspace(0, 1, r)        env[-r:] = np.linspace(1, 0, r)    level[idx] += gain * env * (1 + 0.05 * rng.normal(size=dur_min))    return level

In [ ]:
def make_week(seed=20260827, start="2026-08-17", missing=True):    """One week of L_Aeq,1min for a single room. Tidy long format."""    rng = np.random.default_rng(seed)    minutes = np.arange(MIN_PER_DAY)    frames = []    for d, day in enumerate(DAYS):        weekend = day in ("Saturday", "Sunday")        level = _diurnal_floor(minutes, weekend)        level += rng.normal(0, 0.9)                      # some days are just busier        level += _ar1(MIN_PER_DAY, 0.985, 0.45, rng)     # slow correlated drift        # daytime sources        _add_block(level, int(rng.normal(7*60 + (75 if weekend else 0), 20)),                   int(rng.uniform(25, 45)), rng.uniform(9, 13), rng)      # care round        _add_block(level, int(rng.normal(12*60 + 30, 25)),                   int(rng.uniform(35, 55)), rng.uniform(7, 11), rng)      # lunch        for _ in range(rng.integers(2, 5)):                                # visitors, calls            _add_block(level, int(rng.uniform(9*60, 19*60)),                       int(rng.uniform(10, 40)), rng.uniform(5, 10), rng)        _add_block(level, int(rng.normal(19*60, 40)),                   int(rng.uniform(80, 170)), rng.uniform(8, 14), rng)     # evening TV        if weekend:            _add_block(level, int(rng.normal(15*60, 60)),                       int(rng.uniform(45, 110)), rng.uniform(8, 12), rng) # family visit        # night events - short, loud, and the reason this dataset exists        for _ in range(rng.integers(2, 7)):            s = int(rng.choice([rng.uniform(0, 6.5*60), rng.uniform(23*60, MIN_PER_DAY)]))            _add_block(level, s, int(rng.uniform(1, 5)), rng.uniform(10, 20), rng, ramp=1)        if day == "Thursday":                                  # one disturbed night            for _ in range(9):                _add_block(level, int(rng.uniform(60, 300)),                           int(rng.uniform(1, 6)), rng.uniform(12, 22), rng, ramp=1)        # impulsive floor + sensor noise        spikes = rng.random(MIN_PER_DAY) < 0.015        level[spikes] += rng.gamma(2.0, 3.0, spikes.sum())        level += rng.normal(0, 0.55, MIN_PER_DAY)        level = np.clip(level, 26.0, 95.0)        ts = pd.Timestamp(start) + pd.Timedelta(days=d) + pd.to_timedelta(minutes, "m")        frames.append(pd.DataFrame({"timestamp": ts, "laeq_db": level,                                    "day": day, "minute_of_day": minutes}))    df = pd.concat(frames, ignore_index=True)    if missing:   # non-wear / charging gaps        for day, s, e in [("Wednesday", 13*60 + 20, 14*60 + 35),                          ("Saturday",  10*60,      10*60 + 40)]:            df.loc[(df.day == day) & df.minute_of_day.between(s, e), "laeq_db"] = np.nan    df["laeq_db"] = df["laeq_db"].round(2)    return dfdf = make_week()print(df.shape)df.head()

---## 3. Reshape, and check what you actually haveLong format is right for storage; the chart wants a `1440 × 7` matrix, one column per day. Both the reshape and the coverage check are assertions, not comments — if a day is short or a minute is thinly covered, this should be loud.

In [ ]:
wide = df.pivot_table(index="minute_of_day", columns="day", values="laeq_db")[DAYS]assert wide.shape == (MIN_PER_DAY, 7), f"expected 1440x7, got {wide.shape}"  # DST guardcoverage = wide.notna().sum(axis=1)          # how many days back each minuteMIN_DAYS = 4                                 # below this, the median is not trustworthyprint(f"minutes with full 7-day coverage : {(coverage == 7).sum():>5}")print(f"minutes below the {MIN_DAYS}-day floor    : {(coverage < MIN_DAYS).sum():>5}")print(f"missing samples overall          : {df.laeq_db.isna().sum():>5} "      f"({df.laeq_db.isna().mean():.1%})")

In [ ]:
def smooth(s, win=5):    """Rolling *median*, centred. Suppresses sensor noise without letting a    single 70 dB spike smear across the window the way a rolling mean would."""    return pd.Series(s).rolling(win, center=True, min_periods=1).median()def masked_median(w, min_days=MIN_DAYS):    """Across-day median, blanked wherever too few days back the minute."""    med = w.median(axis=1, skipna=True)    return med.mask(w.notna().sum(axis=1) < min_days)hhmm = FuncFormatter(lambda x, _: f"{int(x) // 60:02d}:{int(x) % 60:02d}")def frame(ax, title, subtitle=None, night=(22 * 60, 7 * 60)):    """Shared furniture: midnight-to-midnight axis, night wash, labels."""    ax.set_xlim(0, MIN_PER_DAY)    ax.xaxis.set_major_locator(MultipleLocator(180))    ax.xaxis.set_major_formatter(hhmm)    ax.axvspan(0, night[1], color=NIGHT, alpha=0.06, lw=0)    ax.axvspan(night[0], MIN_PER_DAY, color=NIGHT, alpha=0.06, lw=0)    ax.set_xlabel("time of day")    ax.set_ylabel(r"$L_{Aeq,1min}$   (dB SPL)")    ax.set_title(title, loc="left", fontweight="semibold")    if subtitle:        ax.text(0, 1.02, subtitle, transform=ax.transAxes, fontsize=8.5,                color="#6B7280", va="bottom")

---## 4. The chart as askedSeven days in soft grey, the across-day median in black, midnight to midnight. Shaded bands are 22:00–07:00.Read it as: the black line is what this room normally does at this time of day; the grey spread is how much that "normally" is worth believing.

In [ ]:
def plot_week(selected=None, win=5, ax=None, show_band=False):    """The requested chart. `selected` highlights one day without hiding the rest."""    if ax is None:        _, ax = plt.subplots(figsize=(13, 5))    if show_band:        lo, hi = smooth(wide.quantile(0.10, axis=1), win), smooth(wide.quantile(0.90, axis=1), win)        ax.fill_between(wide.index, lo, hi, color=GREY, alpha=0.5, lw=0, zorder=0)    else:        for d in DAYS:            ax.plot(wide.index, smooth(wide[d], win), color=GREY,                    lw=0.9, alpha=0.75, zorder=1, solid_capstyle="round")    ax.plot(wide.index, smooth(masked_median(wide), win), color=INK, lw=2.0,            zorder=3, label="median of all days")    if selected:        ax.plot(wide.index, smooth(wide[selected], win), color=HILITE,                lw=1.6, zorder=4, label=selected)    frame(ax, "Ambient sound in one room, midnight to midnight",          f"7 days · 1-minute L_Aeq · {win}-minute rolling median for display")    ax.legend(frameon=False, loc="upper left", fontsize=9)    return axplot_week()plt.tight_layout(); plt.show()

In [ ]:
# the same chart with one day pulled forwardplot_week(selected="Thursday")plt.tight_layout(); plt.show()

Thursday's night is visibly noisier than the black reference between 01:00 and 05:00, while its daytime sits inside the grey. That contrast is only legible *because* the context stayed on the page — filtering to Thursday alone would show a jagged line with nothing to be jagged against.

---## 5. The version I'd actually shipSame data, but the seven grey curves are replaced by two nested percentile bands: 10th–90th and 25th–75th. What you lose is the ability to see individual days as objects. What you gain:- It stays readable at 7 days, 90 days, and 900 days. The spaghetti does not.- The question "is this minute unusual for this room?" becomes a direct visual read — inside the dark band is typical, outside the light band is not.- It shows *where in the day* the days disagree. Here the nights agree to within about 3 dB while the afternoons and evenings scatter over twice that — which is the actual finding, and it is nearly invisible in the spaghetti version.For a room-level baseline, the band is the reference and the selected day is the test against it.

In [ ]:
def plot_band(selected=None, win=5, ax=None):    if ax is None:        _, ax = plt.subplots(figsize=(13, 5))    q = {p: smooth(wide.quantile(p, axis=1), win) for p in (0.10, 0.25, 0.75, 0.90)}    ax.fill_between(wide.index, q[0.10], q[0.90], color=GREY, alpha=0.45, lw=0,                    label="10th–90th percentile")    ax.fill_between(wide.index, q[0.25], q[0.75], color=GREY, alpha=0.85, lw=0,                    label="25th–75th percentile")    ax.plot(wide.index, smooth(masked_median(wide), win), color=INK, lw=2.0, label="median")    if selected:        ax.plot(wide.index, smooth(wide[selected], win), color=HILITE, lw=1.6, label=selected)    frame(ax, "Normal range for this room, by time of day",          "band = spread across the 7 days at each minute")    ax.legend(frameon=False, loc="upper left", fontsize=9, ncols=2)    return axplot_band(selected="Thursday")plt.tight_layout(); plt.show()

---## 6. Choosing a day`ipywidgets` gives a live dropdown. If it isn't installed the cell falls back to a small-multiples grid — seven panels, each with the same black median behind it, which is the printable version of the same idea and is often what you want in a report anyway.```pip install ipywidgets```

In [ ]:
def render(day="Thursday", view="lines", win=5):    fig, ax = plt.subplots(figsize=(13, 5))    sel = None if day == "— none —" else day    (plot_band if view == "band" else plot_week)(selected=sel, win=win, ax=ax)    if sel:                                     # readout: night vs day, this day vs median        night = (wide.index < 7 * 60) | (wide.index >= 22 * 60)        a, b = wide.loc[night, sel].dropna(), masked_median(wide)[night].dropna()        ax.text(0.995, 0.96,                f"{sel} night median  {a.median():.1f} dB\n"                f"week night median  {b.median():.1f} dB\n"                f"difference         {a.median() - b.median():+.1f} dB",                transform=ax.transAxes, ha="right", va="top",                fontsize=9, family="monospace", color="#374151")    plt.tight_layout(); plt.show()try:    import ipywidgets as W    from IPython.display import display    display(W.interactive(        render,        day=W.Dropdown(options=["— none —"] + DAYS, value="Thursday", description="day"),        view=W.ToggleButtons(options=[("all days", "lines"), ("normal range", "band")],                             value="lines", description="context"),        win=W.IntSlider(value=5, min=1, max=31, step=2, description="smooth (min)"),    ))except ImportError:    print("ipywidgets not installed - static small multiples instead\n")    fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True, sharey=True)    med = smooth(masked_median(wide))    for ax, d in zip(axes.ravel(), DAYS):        for o in DAYS:            ax.plot(wide.index, smooth(wide[o]), color=GREY, lw=0.6, alpha=0.5)        ax.plot(wide.index, med, color=INK, lw=1.4)        ax.plot(wide.index, smooth(wide[d]), color=HILITE, lw=1.3)        ax.set_title(d, loc="left", fontsize=10, fontweight="semibold")        ax.set_xlim(0, MIN_PER_DAY)        ax.xaxis.set_major_locator(MultipleLocator(360))        ax.xaxis.set_major_formatter(hhmm)    axes.ravel()[-1].axis("off")    plt.tight_layout(); plt.show()

---## 7. Statistical levels, and why the mean is not the medianAcoustics has its own summary vocabulary, and it's more useful here than mean and standard deviation:- **L90** — the level exceeded 90% of the time. The *background*: how quiet the room gets when nothing is happening. For a sleeping resident this is the number that describes the environment.- **L50** — the median.- **L10** — the level exceeded 10% of the time. The *intrusive* level: how loud the events are. Night-time L10 is the closest single number to "was this night disturbed?".- **LAeq** — the energy-equivalent level, computed in the pressure domain, not on the dB values.Note the gap between `LAeq` and `mean_of_dB` below. It is not a rounding difference — it's the energy of the short loud events, which the arithmetic mean of the dB column throws away. `mean_of_dB` is in the table only as a warning label; never report it.

In [ ]:
def levels(x):    """L90/L50/L10 and the energy-correct LAeq for a block of dB values."""    x = pd.Series(x).dropna()    return pd.Series({        "L90":        x.quantile(0.10),        "L50":        x.quantile(0.50),        "L10":        x.quantile(0.90),        "LAeq":       10 * np.log10(np.mean(10 ** (x / 10))),   # energy domain        "mean_of_dB": x.mean(),                                 # wrong on purpose    })is_night = (wide.index < 7 * 60) | (wide.index >= 22 * 60)night_tbl = pd.DataFrame({d: levels(wide.loc[is_night, d]) for d in DAYS}).Tnight_tbl["LAeq − mean_of_dB"] = night_tbl.LAeq - night_tbl.mean_of_dBnight_tbl.round(1)

In [ ]:
# night L10 as a one-number-per-night summary - the shape a trend line would takeax = night_tbl["L10"].plot(kind="bar", color=GREY, edgecolor=INK, lw=0.8,                           figsize=(9, 3.4), zorder=3)ax.axhline(night_tbl["L10"].median(), color=INK, lw=1.4, ls="--", zorder=4,           label=f"week median  {night_tbl['L10'].median():.1f} dB")ax.set_ylabel("night L10  (dB)")ax.set_xlabel("")ax.set_title("Night-time intrusive level, 22:00–07:00", loc="left", fontweight="semibold")ax.set_ylim(30, night_tbl["L10"].max() + 3)ax.legend(frameon=False, fontsize=9)plt.xticks(rotation=0)plt.tight_layout(); plt.show()

---## 8. Export, and the standalone viewer`sound_week_viewer.html` is the same seven days in a single self-contained file — no Python, no server, no internet after first load. Open it in a browser and it works. It draws the same three inks on a chart-recorder ground, adds a scrubbing cursor that reads out the selected day against the median at any minute, and lets you switch between the spaghetti and band views.That's the artefact to send to a clinician or a partner. This notebook is the artefact for whoever has to trust the numbers.

In [ ]:
import jsonfrom pathlib import Pathout = Path("./outputs"); out.mkdir(exist_ok=True)df.to_csv(out / "sound_week_1min.csv", index=False)payload = {    "days": DAYS,    "series": {d: [None if pd.isna(v) else float(v) for v in wide[d]] for d in DAYS},    "median": [None if pd.isna(v) else round(float(v), 2) for v in masked_median(wide)],    **{k: [round(float(v), 2) for v in wide.quantile(p, axis=1)]       for k, p in [("p10", .10), ("p25", .25), ("p75", .75), ("p90", .90)]},}(out / "sound_week.json").write_text(json.dumps(payload))print("wrote", out / "sound_week_1min.csv", "and", out / "sound_week.json")

---## 9. Noon-to-noon: the frame I'd use for overnight workIdentical data, rolled by 720 minutes. The night now sits whole in the middle of the panel instead of being cut across the two edges, and each grey trace is one continuous *night* rather than two disconnected fragments of two different calendar days.Midnight-to-midnight answers "what happened on Thursday". Noon-to-noon answers "what happened during Thursday night" — and for early detection of overnight deterioration, that's the question.

In [ ]:
rolled = pd.DataFrame({d: np.roll(wide[d].to_numpy(), -720) for d in DAYS})fig, ax = plt.subplots(figsize=(13, 5))for d in DAYS:    ax.plot(rolled.index, smooth(rolled[d]), color=GREY, lw=0.9, alpha=0.75)ax.plot(rolled.index, smooth(masked_median(rolled)), color=INK, lw=2.0, label="median")ax.plot(rolled.index, smooth(rolled["Thursday"]), color=HILITE, lw=1.6, label="Thursday night")ax.axvspan(10 * 60, 19 * 60, color=NIGHT, alpha=0.06, lw=0)     # 22:00 -> 07:00, rolledax.set_xlim(0, MIN_PER_DAY)ax.xaxis.set_major_locator(MultipleLocator(180))ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{(int(x) // 60 + 12) % 24:02d}:00"))ax.set_xlabel("time of day (noon → noon)")ax.set_ylabel(r"$L_{Aeq,1min}$   (dB SPL)")ax.set_title("The same week, sleep-centred", loc="left", fontweight="semibold")ax.legend(frameon=False, loc="upper left", fontsize=9)plt.tight_layout(); plt.show()

---## 10. Where this goes nextThings this notebook sets up but does not do:**Reference bands from more than one week.** The band in §5 is a within-room reference built from 7 days. The same code with a `room_id` grouping produces a between-room reference; the interesting question is which one a given night should be judged against, and the answer is probably both — a night can be normal for the ward and abnormal for the resident.**Event detection rather than level plotting.** Night L10 collapses a night to one number and is a reasonable trend variable, but "12 discrete events" and "one long noisy hour" give the same L10 and mean different things. A segmentation step over the 1-minute series — count, duration, and inter-event interval of excursions above the resident's own L90 + k dB — is a better feature set, and it's a small addition to §7.**Alignment to the other signals.** Sound alone is ambiguous; sound plus heart rate plus actigraphy at the same minute is not. The `minute_of_day` index used throughout is the join key, so the same panel can carry a second axis without restructuring anything.**Honest denominators.** §3 masks minutes below a 4-day floor. Any comparison across weeks needs the same treatment applied to the *week*, not just the minute — a week at 60% coverage should not produce a reference band at all, and that gate belongs upstream of the chart.Everything here is descriptive: it characterises what a room sounds like. Nothing in it is a diagnostic threshold, and the gap between "this night is outside the normal range for this room" and "this resident is deteriorating" is a clinical validation question, not a plotting one.